# 01 · FoodNExTDB download, schema and image audit

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Requires actual supervisor execution and research-use terms decisions. The adapter checks the published schema and stops rather than guessing malformed records.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Check the public-data track and permission records

In [ ]:
from oncoplate.governance import require_gate
assert cfg['study']['dataset']=='foodnextdb', "This notebook is for the separate FoodNExTDB foundation."
require_gate(cfg,'supervisor_execution');require_gate(cfg,'foodnextdb_research_terms')
initialize(cfg)

## 2. Download once, stage locally, audit
The ZIP remains in Drive; extracted small files are read from Colab local storage. The actual SHA-256 is saved; no publisher checksum is invented.

In [ ]:
from oncoplate.pipeline import audit_public
DOWNLOAD_IF_MISSING = True
report=audit_public(cfg,download_missing=DOWNLOAD_IF_MISSING)
print(json.dumps(report,indent=2))

## 3. Inspect actual annotation fields and vocabulary

In [ ]:
records=read_table(p['prepared']/"records.csv")
ann=read_table(p['prepared']/"annotations.csv")
display(records.head());display(ann.head())
print(json.dumps(read_json(p['prepared']/"vocabulary_audit.json"),indent=2))

## 4. Check a real image and its panel records
No bounding boxes or cross-reviewer item identity are inferred.

In [ ]:
from PIL import Image
from IPython.display import display
rid=records.record_id.iloc[0]
display(Image.open(records.loc[records.record_id.eq(rid),'image_path'].iloc[0]).resize((480,480)))
display(ann[ann.record_id.eq(rid)])

## 5. Generate near-duplicate proposals for human review
An edge is joined only when `approved` is explicitly true. Unreviewed similarity is not proof of duplication.

In [ ]:
from oncoplate.splits import propose_near_duplicates
edges=propose_near_duplicates(records,cfg['data']['near_duplicate_hamming_distance'])
review_path=p['prepared']/"near_duplicate_review.csv"
if not review_path.exists():write_table(review_path,edges)
print("Review:",review_path,"| proposed pairs:",len(edges))
print("Use the decision file described in notebook 03 to document completed review; do not split near-duplicate families apart.")

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
